In [ ]:
# ====================================== 导入所需依赖库 ======================================
import sys
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# TensorBoard相关导入（含图表可视化）
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

# 网络结构可视化（优化格式，解决无法打开问题）
from torchviz import make_dot
from torchsummary import summary

# ====================================== 第一步：构建Attention UNet神经网络 ======================================
class ConvBlock(nn.Module):
    """卷积块（双重卷积）：构成网络的基础单元"""
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class EncoderBlock(nn.Module):
    """编码块：下采样+特征提取"""
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        skip = self.conv(x)
        down = self.pool(skip)
        return skip, down

class AttentionGate(nn.Module):
    """注意力门控模块：筛选跳跃连接有效特征"""
    def __init__(self, g_channels, s_channels, out_channels):
        super(AttentionGate, self).__init__()
        self.Wg = nn.Sequential(
            nn.Conv2d(g_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels)
        )
        self.Ws = nn.Sequential(
            nn.Conv2d(s_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(out_channels, 1, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, g, s):
        g1 = self.Wg(g)
        s1 = self.Ws(s)
        out = F.relu(g1 + s1)
        psi = self.psi(out)
        return s * psi

class DecoderBlock(nn.Module):
    """解码块：上采样+特征融合"""
    def __init__(self, in_channels, skip_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.att = AttentionGate(in_channels, skip_channels, out_channels)
        self.conv = ConvBlock(in_channels + skip_channels, out_channels)
    
    def forward(self, x, skip):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=True)
        skip = self.att(x, skip)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class AttentionUNet(nn.Module):
    """Attention UNet完整网络"""
    def __init__(self, in_channels=3, out_channels=1):
        super(AttentionUNet, self).__init__()
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.bottleneck = ConvBlock(256, 512)
        self.dec1 = DecoderBlock(512, 256, 256)
        self.dec2 = DecoderBlock(256, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        s1, p1 = self.enc1(x)
        s2, p2 = self.enc2(p1)
        s3, p3 = self.enc3(p2)
        b1 = self.bottleneck(p3)
        d1 = self.dec1(b1, s3)
        d2 = self.dec2(d1, s2)
        d3 = self.dec3(d2, s1)
        return torch.sigmoid(self.final_conv(d3))

# ====================================== 第二步：新增F1 Score计算函数（二值分割专用） ======================================
def calculate_f1_score(preds, targets, threshold=0.5):
    """
    计算二值分割任务的F1 Score（批量数据）
    Args:
        preds: 模型预测输出（sigmoid后，概率值），shape [B, 1, H, W]
        targets: 真实掩码，shape [B, 1, H, W]
        threshold: 二值化阈值
    Returns:
        f1: 平均F1 Score
        precision: 平均精确率
        recall: 平均召回率
    """
    # 二值化处理（概率值→0/1）
    preds_binary = (preds > threshold).float()
    targets_binary = targets.float()
    
    # 展平张量（方便计算TP/FP/FN）
    preds_flat = preds_binary.view(-1)
    targets_flat = targets_binary.view(-1)
    
    # 计算TP（真阳性）、FP（假阳性）、FN（假阴性）
    tp = (preds_flat * targets_flat).sum().item()
    fp = (preds_flat * (1 - targets_flat)).sum().item()
    fn = ((1 - preds_flat) * targets_flat).sum().item()
    
    # 避免分母为0（添加小epsilon）
    epsilon = 1e-8
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    
    # 计算F1 Score
    f1 = 2 * (precision * recall) / (precision + recall + epsilon)
    
    return f1, precision, recall

# ====================================== 第三步：构建自定义PNG格式数据集类 ======================================
class CustomImageMaskDataset(Dataset):
    """自定义数据集类：加载img/mask目录下的PNG图像"""
    def __init__(self, data_root, transform=None, mask_transform=None):
        super(CustomImageMaskDataset, self).__init__()
        self.images_dir = os.path.join(data_root, "img")
        self.masks_dir = os.path.join(data_root, "mask")
        self.image_filenames = [
            f for f in os.listdir(self.images_dir) 
            if f.lower().endswith('.png') and f.startswith('img_')
        ]
        self.transform = transform
        self.mask_transform = mask_transform
        
        if not os.path.exists(self.images_dir):
            raise FileNotFoundError(f"原始图像文件夹不存在：{self.images_dir}")
        if not os.path.exists(self.masks_dir):
            raise FileNotFoundError(f"掩码图文件夹不存在：{self.masks_dir}")
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        img_name_no_ext = os.path.splitext(img_name)[0]
        
        try:
            img_num = img_name_no_ext.split("_")[1]
        except IndexError:
            raise ValueError(f"文件名格式错误：{img_name}，应为img_数字.png")
        
        mask_name = f"mask_{img_num}.png"
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)
        
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"对应掩码文件不存在：{mask_path}")
        
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        
        if self.transform is not None:
            image = self.transform(image)
        if self.mask_transform is not None:
            mask = self.mask_transform(mask)
        
        return image, mask

# ====================================== 第四步：定义图像/掩码预处理管道 ======================================
image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor()
])

# ====================================== 第五步：训练配置+可视化优化（完整修复版） ======================================
if __name__ == "__main__":
    # ---------------------- 基础配置项 ----------------------
    DATA_ROOT = "./data/ALLIN"
    BATCH_SIZE = 10  # 避免显存不足
    EPOCHS = 100
    LEARNING_RATE = 1e-5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ---------------------- 1. 实例化模型并移至设备 ----------------------
    model = AttentionUNet(in_channels=3, out_channels=1).to(DEVICE)
    print(f"Attention UNet网络实例化成功，使用设备：{DEVICE}")
    
    # ---------------------- 2. 优化网络结构可视化 ----------------------
    try:
        # 生成测试输入张量
        test_input = torch.randn(1, 3, 256, 256).to(DEVICE)
        
        # 方式1：TensorBoard内置Graph可视化
        writer = SummaryWriter("runs/attention_unet_complete")
        writer.add_graph(model, test_input)
        
        # 方式2：生成SVG格式文件
        model_output = model(test_input)
        graph = make_dot(model_output, params=dict(model.named_parameters()))
        graph.render("model_structure", format="svg", directory="./")
        
        # 方式3：打印文本版网络结构
        print("\n========== 网络结构文本汇总 ==========")
        summary(model, input_size=(3, 256, 256))
        
        print("\n网络结构可视化完成：")
        print(f"1. TensorBoard Graphs标签页可查看交互式结构")
        print(f"2. 本地文件已保存为：./model_structure.svg")
    except Exception as e:
        print(f"网络结构可视化部分失败：{e}")
        print("提示：需安装系统级Graphviz工具并配置PATH")
    
    # ---------------------- 3. 实例化数据集和DataLoader ----------------------
    try:
        custom_dataset = CustomImageMaskDataset(
            data_root=DATA_ROOT,
            transform=image_transform,
            mask_transform=mask_transform
        )
        custom_dataloader = DataLoader(
            dataset=custom_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            drop_last=False
        )
        print("\n数据集和DataLoader实例化成功")
    except Exception as e:
        print(f"数据集/DataLoader实例化失败：{e}")
        try:
            writer.close()
        except:
            pass
        sys.exit()
    
    # ---------------------- 4. 配置训练核心组件 ----------------------
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # ---------------------- 5. 初始化指标记录列表 ----------------------
    epoch_losses = []
    epoch_f1_scores = []
    epoch_precisions = []
    
    # ---------------------- 6. 开始训练并记录多指标可视化 ----------------------
    print("=" * 60)
    print(f"开始训练，总轮数：{EPOCHS}，批次大小：{BATCH_SIZE}")
    print("=" * 60)
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        total_f1 = 0.0
        total_precision = 0.0
        
        with tqdm(enumerate(custom_dataloader), total=len(custom_dataloader), desc=f"Epoch {epoch+1}/{EPOCHS}") as pbar:
            for batch_idx, (images, masks) in pbar:
                images = images.to(DEVICE)
                masks = masks.to(DEVICE)
                
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss.backward()
                optimizer.step()
                
                # 计算批次F1 Score
                with torch.no_grad():
                    f1, precision, _ = calculate_f1_score(outputs, masks)
                
                # 累积指标
                total_loss += loss.item()
                total_f1 += f1
                total_precision += precision
                
                # 更新进度条（修复：定义avg_batch_precision）
                avg_batch_loss = total_loss / (batch_idx + 1)
                avg_batch_f1 = total_f1 / (batch_idx + 1)
                avg_batch_precision = total_precision / (batch_idx + 1)
                pbar.set_postfix({
                    "Avg Loss": f"{avg_batch_loss:.6f}",
                    "Avg F1": f"{avg_batch_f1:.6f}",
                    "Avg Precision": f"{avg_batch_precision:.6f}"
                })
        
        # ---------------------- 计算本轮平均指标 ----------------------
        epoch_avg_loss = total_loss / len(custom_dataloader)
        epoch_avg_f1 = total_f1 / len(custom_dataloader)
        epoch_avg_precision = total_precision / len(custom_dataloader)
        
        # ---------------------- 记录指标到列表 ----------------------
        epoch_losses.append(epoch_avg_loss)
        epoch_f1_scores.append(epoch_avg_f1)
        epoch_precisions.append(epoch_avg_precision)
        
        # ---------------------- TensorBoard可视化 ----------------------
        writer.add_scalar("Train/Epoch Average Loss", epoch_avg_loss, epoch + 1)
        writer.add_scalar("Train/Epoch F1 Score", epoch_avg_f1, epoch + 1)
        writer.add_scalar("Train/Epoch Precision", epoch_avg_precision, epoch + 1)
        
        # ---------------------- 打印本轮训练结果 ----------------------
        print(f"\nEpoch {epoch+1}/{EPOCHS} 训练完成：")
        print(f"  平均损失：{epoch_avg_loss:.6f}")
        print(f"  平均F1 Score：{epoch_avg_f1:.6f}")
        print(f"  平均精确率：{epoch_avg_precision:.6f}")
        print("-" * 60)
    
    # ---------------------- 7. 训练完成：本地生成三个独立图表（修复所有错误） ----------------------
    # 动态生成x轴：匹配实际训练轮数（解决维度不匹配）
    epochs_completed = range(1, len(epoch_losses) + 1)
    
    # 图1：训练损失曲线（无意外缩进）
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_completed, epoch_losses, color="red", linewidth=2, label="Average Loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCELoss Value")
    plt.title("Attention UNet Training Loss Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("training_loss_curve.png", dpi=300, bbox_inches="tight")
    plt.close()
    
    # 图2：F1 Score曲线（无意外缩进）
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_completed, epoch_f1_scores, color="blue", linewidth=2, label="F1 Score")
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score Value (0~1)")
    plt.title("Attention UNet Training F1 Score Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("training_f1_curve.png", dpi=300, bbox_inches="tight")
    plt.close()
    
    # 图3：精确率曲线（无意外缩进）
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_completed, epoch_precisions, color="green", linewidth=2, label="Precision")
    plt.xlabel("Epoch")
    plt.ylabel("Precision Value (0~1)")
    plt.title("Attention UNet Training Precision Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("training_precision_curve.png", dpi=300, bbox_inches="tight")
    
    # ---------------------- 8. 收尾：关闭Writer+保存模型 ----------------------
    writer.close()
    torch.save(model.state_dict(), "attention_unet_trained.pth")
    
    print("=" * 60)
    print("训练全部完成！所有结果已保存：")
    print(f"1. 模型权重：attention_unet_trained.pth")
    print(f"2. TensorBoard日志：runs/attention_unet_complete")
    print(f"3. 本地独立图表：training_loss_curve.png、training_f1_curve.png、training_precision_curve.png")
    print(f"4. 网络结构文件：model_structure.svg（需安装Graphviz才能生成）")
